# Random variables as functions, expected value and variance
A random variable maps outcomes to numbers; its distribution gives the expected value (centre) and the variance (spread).

In [ ]:
import itertools
from collections import Counter
from fractions import Fraction
import numpy as np
import plotly.graph_objects as go

## 1. A random variable is a function
Input: one outcome of the sample space. Output: a real number. The rule comes from the event we study.

In [ ]:
# sample space of two dice: 36 ordered pairs
S = list(itertools.product(range(1, 7), repeat=2))
print(len(S), 'outcomes, e.g.', S[:3])

# two random variables on the same sample space
def total(outcome):        # event 'sum is 7' -> look at the sum
    return outcome[0] + outcome[1]

def difference(outcome):   # event 'die 1 beats die 2' -> look at the difference
    return outcome[0] - outcome[1]

for o in [(3, 4), (1, 1), (5, 4), (2, 6)]:
    print(o, '-> sum', total(o), ', difference', difference(o))

In [ ]:
# distribution of the difference D: count the pairs for each value
counts = Counter(difference(o) for o in S)
dist_D = {d: Fraction(counts[d], 36) for d in sorted(counts)}
for d, p in dist_D.items():
    print(d, p)

# event 'die 1 shows more than die 2' = D > 0
print('P(D > 0) =', sum(p for d, p in dist_D.items() if d > 0))

## 2. Expected value
The mean as a weighted average, then $E[X] = \sum x\,P(X = x)$.

In [ ]:
data = [5, 3, 4, 5, 3, 3]
print('plain mean:', sum(data) / len(data))
# each unique value times its share of the data
shares = {v: Fraction(c, len(data)) for v, c in Counter(data).items()}
print('weighted:', float(sum(v * p for v, p in shares.items())))

In [ ]:
# one fair die
die = {x: Fraction(1, 6) for x in range(1, 7)}
E = sum(x * p for x, p in die.items())
print('E[X] =', E, '=', float(E))

# same formula for the difference D and for a coin (head = 1)
print('E[D] =', sum(d * p for d, p in dist_D.items()))
print('E[coin] =', sum(x * p for x, p in {1: Fraction(1, 2), 0: Fraction(1, 2)}.items()))

### The long-run average of 100,000 rolls

In [ ]:
rng = np.random.default_rng(42)          # fixed seed
rolls = rng.integers(1, 7, 100_000)      # 1 to 6
print('average of 100,000 rolls:', rolls.mean())

# running average after each roll (400 log-spaced points plotted)
running = rolls.cumsum() / np.arange(1, rolls.size + 1)
keep = np.unique(np.logspace(0, 5, 400).astype(int)) - 1
fig = go.Figure(go.Scatter(x=keep + 1, y=running[keep], mode='lines'))
fig.add_hline(y=3.5, line_dash='dash')
fig.update_xaxes(type='log', title='number of rolls')
fig.update_yaxes(title='average so far')
fig

## 3. Variance
$\mathrm{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$.

In [ ]:
# definition: expected squared distance from the mean
var_def = sum((x - E) ** 2 * p for x, p in die.items())
# shortcut: E[X^2] minus the square of E[X]
E_X2 = sum(x ** 2 * p for x, p in die.items())
var_short = E_X2 - E ** 2
print('E[X^2] =', E_X2)
print('Var by definition:', var_def, '=', round(float(var_def), 4))
print('Var by shortcut:  ', var_short)
print('standard deviation:', round(float(var_def) ** 0.5, 4))

In [ ]:
# the simulated rolls give nearly the same spread
print('variance of 100,000 rolls:', round(rolls.var(), 4))

## 4. Same mean, different spread
The average of two dice also has mean 3.5, but a smaller variance.

In [ ]:
avg = Counter((a + b) / 2 for a, b in S)
dist_avg = {v: Fraction(c, 36) for v, c in sorted(avg.items())}
E_avg = sum(Fraction(v) * p for v, p in dist_avg.items())
var_avg = sum((Fraction(v) - E_avg) ** 2 * p for v, p in dist_avg.items())
print('E =', E_avg, ' Var =', var_avg, '=', round(float(var_avg), 4))

## 3a. A bet repeated 100 times
37 of 213 people have heard of a film (we lose 1 rupee), 176 have not (we win 1 rupee).

In [ ]:
p_heard = 37 / 213
print(f"P(heard) = {p_heard:.4f} -> rounded {p_heard:.2f}; P(not) = {1 - p_heard:.2f}")
print("100 bets with rounded shares:", round(-1 * 0.17 * 100 + 1 * 0.83 * 100, 2), "rupees; per bet", round(-0.17 + 0.83, 2))
print(f"exact shares: E = {(-37 + 176) / 213:.4f}")
print("second bet (+10 if heard, -1 if not):", round(10 * 0.17 + (-1) * 0.83, 2), f"exact {(10 * 37 - 176) / 213:.4f}")

## 4a. Variance term by term: workouts per week

In [ ]:
x = np.arange(5)
p = np.array([0.1, 0.15, 0.4, 0.25, 0.1])
mu = (x * p).sum()
terms = (x - mu) ** 2 * p
print("mean", round(mu, 2), "terms", terms.round(4), "variance", round(terms.sum(), 2), "sd", round(np.sqrt(terms.sum()), 2))
print("mean +- sd:", round(mu - np.sqrt(terms.sum()), 2), round(mu + np.sqrt(terms.sum()), 2))